# Use regex with file I/O

**The question.** You want to run a pattern over a file — a log, a CSV, a config — and get back the matching lines (or the structured data inside them) without loading the whole thing into memory.

The idiom is `re.compile` once, then stream the file line by line inside a `with` block. Each line is a tiny string the regex can chew through instantly, so the memory footprint stays flat even on multi-gigabyte logs.

In [ ]:
import re
from pathlib import Path

# Create a small demo log so the rest of the cell is self-contained.
sample_path = Path('sample_log.txt')
sample_path.write_text(
    'Server started at 09:00:00\n'
    'User alice logged in at 09:15:30\n'
    'Error: connection timeout at 09:20:45\n'
    'User bob logged in at 09:25:00\n'
    'Warning: high memory usage at 09:30:15\n'
    'Error: disk space low at 10:15:30\n'
)

LOG_PATTERN = re.compile(
    r'(?P<type>Error|Warning|User \w+)'
    r'.*?at\s+'
    r'(?P<time>\d{2}:\d{2}:\d{2})'
)


def parse_log(path: Path) -> list[dict[str, str]]:
    """Stream a log file and return one dict per matched line."""
    entries = []
    with path.open() as fh:
        for line in fh:
            match = LOG_PATTERN.search(line)
            if match:
                entries.append(match.groupdict())
    return entries


for entry in parse_log(sample_path):
    print(f'[{entry["time"]}] {entry["type"]}')

sample_path.unlink()  # tidy up the demo file

## Why it works

Opening a file object in Python and iterating it yields one line at a time. Only the current line sits in memory; the rest stays on disk until the iterator asks for it. That's why this pattern is safe on enormous files.

`re.compile` is called at module import time (or near the top of the function), not inside the loop. Python caches recent patterns, but compiling explicitly makes the intent obvious and guarantees the cost is paid once, not per line.

The `with path.open():` block guarantees the file handle is closed even if the regex or downstream code raises. On Linux that's a nicety; on Windows it's essential, since an open handle prevents the file from being moved or deleted by anything else.

## Trade-offs and when not to use this

- **Small files are easier to search whole.** If the file is under a few megabytes, `path.read_text()` + `re.findall` is shorter and does the job. Use the streaming pattern when size or memory actually matter.
- **Line-by-line fails when matches span multiple lines.** If your pattern needs to see across newlines (e.g. a multi-line stack trace), you have to read the whole file and use `re.DOTALL`, or buffer lines yourself into records first.
- **`re.MULTILINE` is for whole-file searches, not streaming.** The flag only changes what `^` and `$` mean; it doesn't help if you're already iterating line-by-line.
- **Structured formats still deserve their parser.** `json`, `csv`, and `logging`'s own `Formatter` can round-trip a line back to a dict without the fragility of a regex. Use regex when the log format is bespoke and no parser exists.

## Related

- **Recipe** — [Extract data from text](extract-data-from-text.ipynb) for the named-group + `.groupdict()` technique this recipe builds on.
- **Recipe** — [Process a large file lazily](../../iterators-and-generators/recipes/process-a-large-file-lazily.ipynb) for the more general streaming pattern.
- **Reference** — [`re` module quick reference](../reference/re-module-quick-reference.md) and the [File handling guide](../../file-handling/) for `pathlib` basics.
- **Concepts** — [Why context managers matter](../../file-handling/concepts/why-context-managers-matter.md) for the `with` block.